# Mixed Precision Training and Inference Analysis for Thesis Results

This notebook provides comprehensive analysis of mixed-precision training and inference experiments for the thesis Results section. The analysis covers statistical validation, performance optimization, and hardware efficiency across NVIDIA and AMD GPUs.

## Experimental Setup Overview
- **Hardware**: NVIDIA RTX 4090, NVIDIA L40S, AMD RX 7900 XT
- **Models**: BERT-large, GPT-2, ResNet-50
- **Precision Formats**: FP32 (baseline), FP16, BF16
- **Metrics**: Throughput, latency, memory usage, power consumption, GPU utilization

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import glob
from pathlib import Path
from scipy import stats
from scipy.optimize import minimize_scalar
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")
print(f"Seaborn version: {sns.__version__}")

Libraries imported successfully!
NumPy version: 2.0.1
Pandas version: 2.2.2
Matplotlib version: 3.9.2
Seaborn version: 0.13.2


In [15]:
# Data loading and preprocessing functions
def load_inference_data():
    """Load all inference metrics from JSON files"""
    infer_data = []
    
    # Load inference metrics
    infer_metrics_path = Path("infer_metrics/infer_metrics/7900")
    if infer_metrics_path.exists():
        for file_path in infer_metrics_path.glob("*.json"):
            try:
                with open(file_path, 'r') as f:
                    data = json.load(f)
                    
                # Parse filename to extract metadata
                filename = file_path.name
                parts = filename.replace("infer_metrics_", "").replace(".json", "").split("_")
                
                # Extract GPU, model, precision, batch size
                if len(parts) >= 4:
                    gpu = "_".join(parts[:-3])
                    model = parts[-3]
                    precision = parts[-2]
                    batch_size = parts[-1].replace("bs", "")
                    
                    # Create record for each run
                    for i, run_data in enumerate(data.get("runs", [])):
                        record = {
                            'gpu': gpu,
                            'model': model,
                            'precision': precision,
                            'batch_size': int(batch_size),
                            'run_id': i,
                            'throughput_samples_per_sec': run_data.get('throughput_samples_per_sec'),
                            'mean_batch_latency_ms': run_data.get('mean_batch_latency_ms'),
                            'peak_memory_mb': run_data.get('peak_memory_mb'),
                            'gpu_avg_util_percent': run_data.get('gpu_avg_util_percent'),
                            'gpu_avg_power_watts': run_data.get('gpu_avg_power_watts', 0)
                        }
                        infer_data.append(record)
            except Exception as e:
                print(f"Error loading {file_path}: {e}")
    
    return pd.DataFrame(infer_data)

def load_training_data():
    """Load training metrics from CSV files"""
    train_data = []
    
    # Load from results folder
    results_folders = ['results/nvidia_runs', 'results/amd_runs', 'results/ai_nvidia_runs']
    
    for folder in results_folders:
        folder_path = Path(folder)
        if folder_path.exists():
            for file_path in folder_path.glob("train_*.csv"):
                try:
                    df = pd.read_csv(file_path)
                    
                    # Parse filename to extract metadata
                    filename = file_path.name
                    parts = filename.replace("train_", "").replace(".csv", "").split("_")
                    
                    if len(parts) >= 5:
                        model = parts[0]
                        precision = parts[1]
                        batch_size = parts[2].replace("bs", "")
                        epochs = parts[3].replace("e", "")
                        run_id = parts[4].replace("r", "")
                        
                        # Determine GPU from folder
                        if 'nvidia' in folder:
                            gpu = 'NVIDIA_RTX_4090' if '4090' in folder else 'NVIDIA_L40S'
                        else:
                            gpu = 'AMD_RX_7900_XT'
                        
                        # Add metadata to dataframe
                        df['gpu'] = gpu
                        df['model'] = model
                        df['precision'] = precision
                        df['batch_size'] = int(batch_size)
                        df['total_epochs'] = int(epochs)
                        df['run_id'] = int(run_id)
                        
                        train_data.append(df)
                        
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")
    
    if train_data:
        return pd.concat(train_data, ignore_index=True)
    else:
        return pd.DataFrame()

# Load the datasets
print("Loading inference data...")
inference_df = load_inference_data()
print(f"Loaded {len(inference_df)} inference records")

print("Loading training data...")
training_df = load_training_data()
print(f"Loaded {len(training_df)} training records")

# Display data overview
if not inference_df.empty:
    print("\n=== INFERENCE DATA OVERVIEW ===")
    print(f"Shape: {inference_df.shape}")
    print(f"GPUs: {inference_df['gpu'].unique()}")
    print(f"Models: {inference_df['model'].unique()}")
    print(f"Precisions: {inference_df['precision'].unique()}")
    print(f"Batch sizes: {sorted(inference_df['batch_size'].unique())}")

if not training_df.empty:
    print("\n=== TRAINING DATA OVERVIEW ===")
    print(f"Shape: {training_df.shape}")
    print(f"GPUs: {training_df['gpu'].unique()}")
    print(f"Models: {training_df['model'].unique()}")
    print(f"Precisions: {training_df['precision'].unique()}")

Loading inference data...
Loaded 0 inference records
Loading training data...
Error loading results/ai_nvidia_runs/train_resnet50_fp16_bs256_e30_r1_raw_metrics.csv: No columns to parse from file
Error loading results/ai_nvidia_runs/train_resnet50_fp32_bs128_e30_r1_raw_metrics.csv: No columns to parse from file
Error loading results/ai_nvidia_runs/train_resnet50_bf16_bs256_e30_r1_raw_metrics.csv: No columns to parse from file
Loaded 30097 training records

=== TRAINING DATA OVERVIEW ===
Shape: (30097, 21)
GPUs: ['NVIDIA_L40S' 'AMD_RX_7900_XT']
Models: ['gpt2' 'bert-large' 'resnet50']
Precisions: ['bf16' 'fp32' 'fp16']
Error loading results/ai_nvidia_runs/train_resnet50_fp16_bs256_e30_r1_raw_metrics.csv: No columns to parse from file
Error loading results/ai_nvidia_runs/train_resnet50_fp32_bs128_e30_r1_raw_metrics.csv: No columns to parse from file
Error loading results/ai_nvidia_runs/train_resnet50_bf16_bs256_e30_r1_raw_metrics.csv: No columns to parse from file
Loaded 30097 training re

## 1. Statistical Analysis and Hypothesis Testing

This section performs rigorous statistical tests to validate performance differences between precision formats. We use t-tests, ANOVA, and confidence intervals to establish statistical significance of observed performance improvements.

In [3]:
def perform_statistical_analysis(df, metric_column, group_columns):
    """Perform comprehensive statistical analysis on performance metrics"""
    
    results = {}
    
    # One-way ANOVA for each grouping
    for group_col in group_columns:
        groups = []
        group_names = []
        
        for group_name in df[group_col].unique():
            group_data = df[df[group_col] == group_name][metric_column].dropna()
            if len(group_data) > 0:
                groups.append(group_data)
                group_names.append(group_name)
        
        if len(groups) >= 2:
            f_stat, p_value = stats.f_oneway(*groups)
            results[f'ANOVA_{group_col}'] = {
                'f_statistic': f_stat,
                'p_value': p_value,
                'significant': p_value < 0.05
            }
    
    return results

def pairwise_t_tests(df, metric_column, precision_col='precision'):
    """Perform pairwise t-tests between precision formats"""
    precisions = df[precision_col].unique()
    results = {}
    
    for i, prec1 in enumerate(precisions):
        for j, prec2 in enumerate(precisions):
            if i < j:  # Avoid duplicate comparisons
                group1 = df[df[precision_col] == prec1][metric_column].dropna()
                group2 = df[df[precision_col] == prec2][metric_column].dropna()
                
                if len(group1) > 1 and len(group2) > 1:
                    # Independent t-test
                    t_stat, p_value = stats.ttest_ind(group1, group2)
                    
                    # Effect size (Cohen's d)
                    pooled_std = np.sqrt(((len(group1)-1)*group1.std()**2 + 
                                        (len(group2)-1)*group2.std()**2) / 
                                       (len(group1)+len(group2)-2))
                    cohens_d = (group1.mean() - group2.mean()) / pooled_std
                    
                    results[f'{prec1}_vs_{prec2}'] = {
                        't_statistic': t_stat,
                        'p_value': p_value,
                        'cohens_d': cohens_d,
                        'significant': p_value < 0.05,
                        'effect_size': 'small' if abs(cohens_d) < 0.5 else 'medium' if abs(cohens_d) < 0.8 else 'large'
                    }
    
    return results

def calculate_confidence_intervals(df, metric_column, group_col, confidence=0.95):
    """Calculate confidence intervals for each group"""
    alpha = 1 - confidence
    results = {}
    
    for group in df[group_col].unique():
        data = df[df[group_col] == group][metric_column].dropna()
        
        if len(data) > 1:
            mean = data.mean()
            sem = stats.sem(data)  # Standard error of mean
            ci = stats.t.interval(confidence, len(data)-1, loc=mean, scale=sem)
            
            results[group] = {
                'mean': mean,
                'std': data.std(),
                'sem': sem,
                'ci_lower': ci[0],
                'ci_upper': ci[1],
                'n': len(data)
            }
    
    return results

# Perform statistical analysis on inference data
if not inference_df.empty:
    print("=== STATISTICAL ANALYSIS ON INFERENCE THROUGHPUT ===")
    
    # ANOVA tests
    throughput_anova = perform_statistical_analysis(
        inference_df, 'throughput_samples_per_sec', 
        ['precision', 'gpu', 'model']
    )
    
    for test, result in throughput_anova.items():
        significance = "SIGNIFICANT" if result['significant'] else "NOT SIGNIFICANT"
        print(f"{test}: F={result['f_statistic']:.3f}, p={result['p_value']:.6f} ({significance})")
    
    print("\n=== PAIRWISE T-TESTS FOR PRECISION FORMATS ===")
    
    # T-tests between precision formats
    precision_ttests = pairwise_t_tests(inference_df, 'throughput_samples_per_sec')
    
    for comparison, result in precision_ttests.items():
        significance = "SIGNIFICANT" if result['significant'] else "NOT SIGNIFICANT"
        effect = result['effect_size'].upper()
        print(f"{comparison}: t={result['t_statistic']:.3f}, p={result['p_value']:.6f}, "
              f"Cohen's d={result['cohens_d']:.3f} ({effect} effect, {significance})")
    
    print("\n=== CONFIDENCE INTERVALS (95%) FOR THROUGHPUT BY PRECISION ===")
    
    # Confidence intervals
    precision_ci = calculate_confidence_intervals(
        inference_df, 'throughput_samples_per_sec', 'precision'
    )
    
    for precision, stats_info in precision_ci.items():
        print(f"{precision}: {stats_info['mean']:.2f} ± {stats_info['sem']:.2f} samples/sec "
              f"[95% CI: {stats_info['ci_lower']:.2f}, {stats_info['ci_upper']:.2f}] (n={stats_info['n']})")

else:
    print("No inference data available for statistical analysis")

No inference data available for statistical analysis


In [4]:
# Visualization of statistical results
def plot_statistical_comparison(df, metric_column, group_col, title):
    """Create comprehensive statistical visualization"""
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # Box plot with statistical annotations
    sns.boxplot(data=df, x=group_col, y=metric_column, ax=ax1)
    ax1.set_title(f'{title} - Distribution by {group_col}')
    ax1.tick_params(axis='x', rotation=45)
    
    # Violin plot for distribution shape
    sns.violinplot(data=df, x=group_col, y=metric_column, ax=ax2)
    ax2.set_title(f'{title} - Density Distribution')
    ax2.tick_params(axis='x', rotation=45)
    
    # Mean with confidence intervals
    ci_data = calculate_confidence_intervals(df, metric_column, group_col)
    groups = list(ci_data.keys())
    means = [ci_data[g]['mean'] for g in groups]
    ci_lower = [ci_data[g]['ci_lower'] for g in groups]
    ci_upper = [ci_data[g]['ci_upper'] for g in groups]
    
    ax3.errorbar(range(len(groups)), means, 
                yerr=[np.array(means) - np.array(ci_lower), 
                      np.array(ci_upper) - np.array(means)], 
                fmt='o', capsize=5, capthick=2, markersize=8)
    ax3.set_xticks(range(len(groups)))
    ax3.set_xticklabels(groups, rotation=45)
    ax3.set_title(f'{title} - Means with 95% Confidence Intervals')
    ax3.grid(True, alpha=0.3)
    
    # Effect size heatmap (if multiple precisions)
    if group_col == 'precision' and len(groups) > 1:
        ttests = pairwise_t_tests(df, metric_column)
        effect_matrix = np.zeros((len(groups), len(groups)))
        
        for i, g1 in enumerate(groups):
            for j, g2 in enumerate(groups):
                if i != j:
                    key = f'{g1}_vs_{g2}' if f'{g1}_vs_{g2}' in ttests else f'{g2}_vs_{g1}'
                    if key in ttests:
                        effect_matrix[i, j] = ttests[key]['cohens_d']
        
        im = ax4.imshow(effect_matrix, cmap='RdBu_r', vmin=-2, vmax=2)
        ax4.set_xticks(range(len(groups)))
        ax4.set_yticks(range(len(groups)))
        ax4.set_xticklabels(groups)
        ax4.set_yticklabels(groups)
        ax4.set_title("Effect Size Matrix (Cohen's d)")
        
        # Add text annotations
        for i in range(len(groups)):
            for j in range(len(groups)):
                ax4.text(j, i, f'{effect_matrix[i, j]:.2f}', 
                        ha='center', va='center', color='white' if abs(effect_matrix[i, j]) > 1 else 'black')
        
        plt.colorbar(im, ax=ax4)
    else:
        ax4.axis('off')
    
    plt.tight_layout()
    plt.show()

# Create statistical visualizations
if not inference_df.empty:
    print("Creating statistical visualizations...")
    
    # Throughput analysis by precision
    plot_statistical_comparison(
        inference_df, 'throughput_samples_per_sec', 'precision', 
        'Inference Throughput Statistical Analysis'
    )
    
    # Latency analysis by GPU
    plot_statistical_comparison(
        inference_df, 'mean_batch_latency_ms', 'gpu', 
        'Batch Latency Statistical Analysis'
    )
    
    # Memory usage by model
    plot_statistical_comparison(
        inference_df, 'peak_memory_mb', 'model', 
        'Peak Memory Usage Statistical Analysis'
    )

## 2. Memory Usage Analysis

This section analyzes memory consumption patterns across different precisions and batch sizes, including peak memory usage, memory efficiency ratios, and memory scaling behavior.

In [5]:
def analyze_memory_usage(df):
    """Comprehensive memory usage analysis"""
    
    if 'peak_memory_mb' not in df.columns:
        print("Memory data not available")
        return
    
    # Calculate memory efficiency metrics
    memory_analysis = df.groupby(['gpu', 'model', 'precision']).agg({
        'peak_memory_mb': ['mean', 'std', 'min', 'max'],
        'throughput_samples_per_sec': 'mean',
        'batch_size': 'mean'
    }).round(2)
    
    memory_analysis.columns = ['_'.join(col).strip() for col in memory_analysis.columns]
    memory_analysis = memory_analysis.reset_index()
    
    # Calculate memory efficiency (throughput per MB)
    memory_analysis['memory_efficiency'] = (
        memory_analysis['throughput_samples_per_sec_mean'] / 
        memory_analysis['peak_memory_mb_mean']
    )
    
    # Calculate memory savings compared to FP32
    fp32_memory = memory_analysis[memory_analysis['precision'] == 'prfp32'].groupby(['gpu', 'model'])['peak_memory_mb_mean'].first()
    
    memory_savings = []
    for _, row in memory_analysis.iterrows():
        gpu, model, precision = row['gpu'], row['model'], row['precision']
        current_memory = row['peak_memory_mb_mean']
        
        if precision != 'prfp32' and (gpu, model) in fp32_memory.index:
            baseline_memory = fp32_memory[(gpu, model)]
            savings = ((baseline_memory - current_memory) / baseline_memory) * 100
            memory_savings.append(savings)
        else:
            memory_savings.append(0.0)
    
    memory_analysis['memory_savings_percent'] = memory_savings
    
    return memory_analysis

def plot_memory_analysis(memory_df, inference_df):
    """Create comprehensive memory usage visualizations"""
    
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    
    # 1. Memory usage by precision
    sns.barplot(data=memory_df, x='precision', y='peak_memory_mb_mean', 
                hue='model', ax=axes[0,0])
    axes[0,0].set_title('Peak Memory Usage by Precision Format')
    axes[0,0].set_ylabel('Peak Memory (MB)')
    axes[0,0].legend(title='Model')
    
    # 2. Memory efficiency (throughput per MB)
    sns.barplot(data=memory_df, x='precision', y='memory_efficiency', 
                hue='gpu', ax=axes[0,1])
    axes[0,1].set_title('Memory Efficiency (Throughput/Memory)')
    axes[0,1].set_ylabel('Samples/sec per MB')
    axes[0,1].legend(title='GPU', bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # 3. Memory savings compared to FP32
    memory_savings_data = memory_df[memory_df['precision'] != 'prfp32']
    sns.barplot(data=memory_savings_data, x='precision', y='memory_savings_percent', 
                hue='model', ax=axes[0,2])
    axes[0,2].set_title('Memory Savings vs FP32')
    axes[0,2].set_ylabel('Memory Savings (%)')
    axes[0,2].legend(title='Model')
    
    # 4. Memory scaling with batch size
    if not inference_df.empty:
        # Calculate memory per sample
        inference_df['memory_per_sample'] = inference_df['peak_memory_mb'] / inference_df['batch_size']
        
        sns.scatterplot(data=inference_df, x='batch_size', y='peak_memory_mb', 
                       hue='precision', style='model', s=100, ax=axes[1,0])
        axes[1,0].set_title('Memory Scaling with Batch Size')
        axes[1,0].set_xlabel('Batch Size')
        axes[1,0].set_ylabel('Peak Memory (MB)')
    
    # 5. Memory distribution by GPU
    sns.boxplot(data=memory_df, x='gpu', y='peak_memory_mb_mean', ax=axes[1,1])
    axes[1,1].set_title('Memory Usage Distribution by GPU')
    axes[1,1].set_ylabel('Peak Memory (MB)')
    axes[1,1].tick_params(axis='x', rotation=45)
    
    # 6. Memory variance analysis
    sns.scatterplot(data=memory_df, x='peak_memory_mb_mean', y='peak_memory_mb_std', 
                   hue='precision', size='throughput_samples_per_sec_mean', 
                   sizes=(50, 200), ax=axes[1,2])
    axes[1,2].set_title('Memory Usage Variance vs Mean')
    axes[1,2].set_xlabel('Mean Peak Memory (MB)')
    axes[1,2].set_ylabel('Memory Std Dev (MB)')
    
    plt.tight_layout()
    plt.show()

# Perform memory analysis
if not inference_df.empty:
    print("=== MEMORY USAGE ANALYSIS ===")
    
    memory_analysis_results = analyze_memory_usage(inference_df)
    
    if memory_analysis_results is not None:
        print("\nMemory Usage Summary by Configuration:")
        print(memory_analysis_results[['gpu', 'model', 'precision', 'peak_memory_mb_mean', 
                                     'memory_efficiency', 'memory_savings_percent']].to_string(index=False))
        
        print("\n=== KEY MEMORY INSIGHTS ===")
        
        # Find most memory efficient configurations
        most_efficient = memory_analysis_results.nlargest(3, 'memory_efficiency')
        print("\nMost Memory Efficient Configurations:")
        for _, row in most_efficient.iterrows():
            print(f"  {row['gpu']} + {row['model']} + {row['precision']}: "
                  f"{row['memory_efficiency']:.3f} samples/sec/MB")
        
        # Find highest memory savings
        highest_savings = memory_analysis_results[memory_analysis_results['precision'] != 'prfp32'].nlargest(3, 'memory_savings_percent')
        print("\nHighest Memory Savings vs FP32:")
        for _, row in highest_savings.iterrows():
            print(f"  {row['gpu']} + {row['model']} + {row['precision']}: "
                  f"{row['memory_savings_percent']:.1f}% savings")
        
        # Create visualizations
        print("\nCreating memory usage visualizations...")
        plot_memory_analysis(memory_analysis_results, inference_df)
        
        # Memory scaling analysis
        print("\n=== MEMORY SCALING ANALYSIS ===")
        for precision in inference_df['precision'].unique():
            precision_data = inference_df[inference_df['precision'] == precision]
            if len(precision_data) > 1:
                correlation = precision_data['batch_size'].corr(precision_data['peak_memory_mb'])
                print(f"{precision}: Batch Size vs Memory correlation = {correlation:.3f}")
else:
    print("No inference data available for memory analysis")

No inference data available for memory analysis


## 3. Energy Efficiency Comparison

This section calculates and compares energy consumption metrics (performance per watt) across different hardware and precision combinations to identify the most energy-efficient configurations.

In [6]:
def analyze_energy_efficiency(df):
    """Analyze energy efficiency metrics"""
    
    if 'gpu_avg_power_watts' not in df.columns:
        print("Power consumption data not available")
        return None
    
    # Filter out zero power readings
    power_data = df[df['gpu_avg_power_watts'] > 0].copy()
    
    if power_data.empty:
        print("No valid power consumption data found")
        return None
    
    # Calculate energy efficiency metrics
    power_data['performance_per_watt'] = power_data['throughput_samples_per_sec'] / power_data['gpu_avg_power_watts']
    power_data['energy_per_sample'] = power_data['gpu_avg_power_watts'] / power_data['throughput_samples_per_sec']
    
    # Aggregate by configuration
    energy_analysis = power_data.groupby(['gpu', 'model', 'precision']).agg({
        'gpu_avg_power_watts': ['mean', 'std'],
        'throughput_samples_per_sec': 'mean',
        'performance_per_watt': ['mean', 'std'],
        'energy_per_sample': ['mean', 'std']
    }).round(3)
    
    energy_analysis.columns = ['_'.join(col).strip() for col in energy_analysis.columns]
    energy_analysis = energy_analysis.reset_index()
    
    # Calculate energy savings compared to FP32
    fp32_energy = energy_analysis[energy_analysis['precision'] == 'prfp32'].groupby(['gpu', 'model'])['energy_per_sample_mean'].first()
    
    energy_savings = []
    for _, row in energy_analysis.iterrows():
        gpu, model, precision = row['gpu'], row['model'], row['precision']
        current_energy = row['energy_per_sample_mean']
        
        if precision != 'prfp32' and (gpu, model) in fp32_energy.index:
            baseline_energy = fp32_energy[(gpu, model)]
            savings = ((baseline_energy - current_energy) / baseline_energy) * 100
            energy_savings.append(savings)
        else:
            energy_savings.append(0.0)
    
    energy_analysis['energy_savings_percent'] = energy_savings
    
    return energy_analysis

def plot_energy_analysis(energy_df, power_df):
    """Create comprehensive energy efficiency visualizations"""
    
    if energy_df is None:
        print("No energy data to plot")
        return
    
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    
    # 1. Performance per Watt by precision
    sns.barplot(data=energy_df, x='precision', y='performance_per_watt_mean', 
                hue='model', ax=axes[0,0])
    axes[0,0].set_title('Performance per Watt by Precision')
    axes[0,0].set_ylabel('Samples/sec/Watt')
    axes[0,0].legend(title='Model')
    
    # 2. Power consumption by GPU
    sns.barplot(data=energy_df, x='gpu', y='gpu_avg_power_watts_mean', 
                hue='precision', ax=axes[0,1])
    axes[0,1].set_title('Average Power Consumption by GPU')
    axes[0,1].set_ylabel('Power (Watts)')
    axes[0,1].tick_params(axis='x', rotation=45)
    axes[0,1].legend(title='Precision')
    
    # 3. Energy per sample
    sns.barplot(data=energy_df, x='precision', y='energy_per_sample_mean', 
                hue='gpu', ax=axes[0,2])
    axes[0,2].set_title('Energy per Sample by Precision')
    axes[0,2].set_ylabel('Watts/Sample')
    axes[0,2].legend(title='GPU', bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # 4. Energy efficiency scatter
    sns.scatterplot(data=energy_df, x='gpu_avg_power_watts_mean', y='performance_per_watt_mean', 
                   hue='precision', style='model', s=150, ax=axes[1,0])
    axes[1,0].set_title('Energy Efficiency vs Power Consumption')
    axes[1,0].set_xlabel('Average Power (Watts)')
    axes[1,0].set_ylabel('Performance/Watt')
    
    # 5. Energy savings compared to FP32
    energy_savings_data = energy_df[energy_df['precision'] != 'prfp32']
    if not energy_savings_data.empty:
        sns.barplot(data=energy_savings_data, x='precision', y='energy_savings_percent', 
                   hue='model', ax=axes[1,1])
        axes[1,1].set_title('Energy Savings vs FP32')
        axes[1,1].set_ylabel('Energy Savings (%)')
        axes[1,1].legend(title='Model')
    
    # 6. Power efficiency heatmap
    pivot_data = energy_df.pivot_table(values='performance_per_watt_mean', 
                                      index='model', columns='precision', aggfunc='mean')
    sns.heatmap(pivot_data, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[1,2])
    axes[1,2].set_title('Performance/Watt Heatmap')
    
    plt.tight_layout()
    plt.show()

# Perform energy efficiency analysis
if not inference_df.empty:
    print("=== ENERGY EFFICIENCY ANALYSIS ===")
    
    energy_results = analyze_energy_efficiency(inference_df)
    
    if energy_results is not None:
        print("\nEnergy Efficiency Summary:")
        display_cols = ['gpu', 'model', 'precision', 'gpu_avg_power_watts_mean', 
                       'performance_per_watt_mean', 'energy_per_sample_mean', 'energy_savings_percent']
        print(energy_results[display_cols].to_string(index=False))
        
        print("\n=== KEY ENERGY INSIGHTS ===")
        
        # Most energy efficient configurations
        most_efficient = energy_results.nlargest(5, 'performance_per_watt_mean')
        print("\nMost Energy Efficient Configurations (Performance/Watt):")
        for _, row in most_efficient.iterrows():
            print(f"  {row['gpu']} + {row['model']} + {row['precision']}: "
                  f"{row['performance_per_watt_mean']:.3f} samples/sec/Watt")
        
        # Highest energy savings
        if 'energy_savings_percent' in energy_results.columns:
            highest_savings = energy_results[energy_results['precision'] != 'prfp32'].nlargest(3, 'energy_savings_percent')
            print("\nHighest Energy Savings vs FP32:")
            for _, row in highest_savings.iterrows():
                print(f"  {row['gpu']} + {row['model']} + {row['precision']}: "
                      f"{row['energy_savings_percent']:.1f}% energy savings")
        
        # Power consumption comparison
        print("\n=== POWER CONSUMPTION COMPARISON ===")
        gpu_power = energy_results.groupby('gpu')['gpu_avg_power_watts_mean'].mean().sort_values(ascending=False)
        print("Average Power Consumption by GPU:")
        for gpu, power in gpu_power.items():
            print(f"  {gpu}: {power:.1f} Watts")
        
        # Create energy visualizations
        print("\nCreating energy efficiency visualizations...")
        plot_energy_analysis(energy_results, inference_df)
        
        # Energy efficiency rankings
        print("\n=== ENERGY EFFICIENCY RANKINGS ===")
        ranking = energy_results.nlargest(10, 'performance_per_watt_mean')
        for i, (_, row) in enumerate(ranking.iterrows(), 1):
            print(f"{i:2d}. {row['gpu']} + {row['model']} + {row['precision']}: "
                  f"{row['performance_per_watt_mean']:.3f} samples/sec/Watt")
    
    else:
        print("Energy analysis could not be performed due to missing power data")
else:
    print("No inference data available for energy analysis")

No inference data available for energy analysis


## 4. Model-Specific Performance Patterns

This section provides a deep dive into how different model architectures (BERT, GPT-2, ResNet) respond to mixed-precision training, identifying architecture-specific optimization opportunities.

In [7]:
def analyze_model_specific_patterns(df):
    """Analyze performance patterns specific to each model architecture"""
    
    model_analysis = {}
    
    for model in df['model'].unique():
        model_data = df[df['model'] == model]
        
        # Performance improvement analysis
        performance_by_precision = model_data.groupby('precision').agg({
            'throughput_samples_per_sec': ['mean', 'std', 'count'],
            'mean_batch_latency_ms': ['mean', 'std'],
            'peak_memory_mb': ['mean', 'std'],
            'gpu_avg_util_percent': ['mean', 'std'] if 'gpu_avg_util_percent' in model_data.columns else ['count']
        }).round(3)
        
        # Calculate improvement ratios vs FP32
        fp32_throughput = model_data[model_data['precision'] == 'prfp32']['throughput_samples_per_sec'].mean()
        
        improvement_ratios = {}
        for precision in model_data['precision'].unique():
            if precision != 'prfp32':
                precision_throughput = model_data[model_data['precision'] == precision]['throughput_samples_per_sec'].mean()
                if fp32_throughput > 0:
                    improvement_ratios[precision] = precision_throughput / fp32_throughput
        
        # GPU utilization efficiency
        gpu_efficiency = model_data.groupby(['gpu', 'precision']).agg({
            'gpu_avg_util_percent': 'mean' if 'gpu_avg_util_percent' in model_data.columns else lambda x: 0,
            'throughput_samples_per_sec': 'mean'
        })
        
        model_analysis[model] = {
            'performance_stats': performance_by_precision,
            'improvement_ratios': improvement_ratios,
            'gpu_efficiency': gpu_efficiency
        }
    
    return model_analysis

def plot_model_specific_analysis(df):
    """Create model-specific performance visualizations"""
    
    models = df['model'].unique()
    n_models = len(models)
    
    fig, axes = plt.subplots(2, n_models, figsize=(6*n_models, 12))
    if n_models == 1:
        axes = axes.reshape(-1, 1)
    
    # Performance comparison for each model
    for i, model in enumerate(models):
        model_data = df[df['model'] == model]
        
        # Throughput by precision
        sns.boxplot(data=model_data, x='precision', y='throughput_samples_per_sec', ax=axes[0, i])
        axes[0, i].set_title(f'{model} - Throughput by Precision')
        axes[0, i].set_ylabel('Throughput (samples/sec)')
        
        # Memory vs Performance scatter
        sns.scatterplot(data=model_data, x='peak_memory_mb', y='throughput_samples_per_sec', 
                       hue='precision', style='gpu', s=100, ax=axes[1, i])
        axes[1, i].set_title(f'{model} - Memory vs Performance')
        axes[1, i].set_xlabel('Peak Memory (MB)')
        axes[1, i].set_ylabel('Throughput (samples/sec)')
    
    plt.tight_layout()
    plt.show()
    
    # Create improvement ratios visualization
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    
    improvement_data = []
    for model in models:
        model_data = df[df['model'] == model]
        fp32_throughput = model_data[model_data['precision'] == 'prfp32']['throughput_samples_per_sec'].mean()
        
        for precision in ['prbf16', 'prfp16']:
            if precision in model_data['precision'].values:
                precision_throughput = model_data[model_data['precision'] == precision]['throughput_samples_per_sec'].mean()
                if fp32_throughput > 0:
                    improvement = ((precision_throughput - fp32_throughput) / fp32_throughput) * 100
                    improvement_data.append({
                        'model': model,
                        'precision': precision,
                        'improvement_percent': improvement
                    })
    
    if improvement_data:
        improvement_df = pd.DataFrame(improvement_data)
        sns.barplot(data=improvement_df, x='model', y='improvement_percent', hue='precision', ax=ax)
        ax.set_title('Performance Improvement vs FP32 by Model')
        ax.set_ylabel('Performance Improvement (%)')
        ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

def create_model_efficiency_heatmap(df):
    """Create efficiency heatmap for model-gpu-precision combinations"""
    
    # Calculate efficiency metric (throughput per memory)
    df_efficiency = df.copy()
    df_efficiency['efficiency'] = df_efficiency['throughput_samples_per_sec'] / df_efficiency['peak_memory_mb']
    
    # Create pivot table
    efficiency_pivot = df_efficiency.pivot_table(
        values='efficiency', 
        index=['model', 'gpu'], 
        columns='precision', 
        aggfunc='mean'
    )
    
    # Create heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(efficiency_pivot, annot=True, fmt='.3f', cmap='YlOrRd', 
                cbar_kws={'label': 'Efficiency (samples/sec/MB)'})
    plt.title('Model-GPU-Precision Efficiency Matrix')
    plt.ylabel('Model + GPU')
    plt.xlabel('Precision Format')
    plt.tight_layout()
    plt.show()
    
    return efficiency_pivot

# Perform model-specific analysis
if not inference_df.empty:
    print("=== MODEL-SPECIFIC PERFORMANCE ANALYSIS ===")
    
    model_patterns = analyze_model_specific_patterns(inference_df)
    
    # Display analysis for each model
    for model, analysis in model_patterns.items():
        print(f"\n--- {model.upper()} MODEL ANALYSIS ---")
        
        print(f"\nPerformance Statistics:")
        print(analysis['performance_stats'])
        
        if analysis['improvement_ratios']:
            print(f"\nPerformance Improvement vs FP32:")
            for precision, ratio in analysis['improvement_ratios'].items():
                improvement = (ratio - 1) * 100
                print(f"  {precision}: {improvement:+.1f}% ({ratio:.2f}x)")
        
        print(f"\nGPU Efficiency by Configuration:")
        print(analysis['gpu_efficiency'])
    
    # Model architecture insights
    print("\n=== MODEL ARCHITECTURE INSIGHTS ===")
    
    # Calculate model-specific metrics
    model_summary = inference_df.groupby('model').agg({
        'throughput_samples_per_sec': ['mean', 'std'],
        'peak_memory_mb': ['mean', 'std'],
        'mean_batch_latency_ms': ['mean', 'std']
    }).round(2)
    
    print("\nModel Performance Summary:")
    print(model_summary)
    
    # Find best configurations for each model
    print("\n=== OPTIMAL CONFIGURATIONS BY MODEL ===")
    for model in inference_df['model'].unique():
        model_data = inference_df[inference_df['model'] == model]
        best_config = model_data.loc[model_data['throughput_samples_per_sec'].idxmax()]
        
        print(f"\n{model} - Best Configuration:")
        print(f"  GPU: {best_config['gpu']}")
        print(f"  Precision: {best_config['precision']}")
        print(f"  Batch Size: {best_config['batch_size']}")
        print(f"  Throughput: {best_config['throughput_samples_per_sec']:.2f} samples/sec")
        print(f"  Memory: {best_config['peak_memory_mb']:.1f} MB")
        print(f"  Latency: {best_config['mean_batch_latency_ms']:.2f} ms")
    
    # Create visualizations
    print("\nCreating model-specific visualizations...")
    plot_model_specific_analysis(inference_df)
    
    print("\nCreating efficiency heatmap...")
    efficiency_matrix = create_model_efficiency_heatmap(inference_df)
    
    # Architecture-specific recommendations
    print("\n=== ARCHITECTURE-SPECIFIC RECOMMENDATIONS ===")
    
    for model in inference_df['model'].unique():
        model_data = inference_df[inference_df['model'] == model]
        
        # Find best precision for each GPU
        best_by_gpu = model_data.loc[model_data.groupby('gpu')['throughput_samples_per_sec'].idxmax()]
        
        print(f"\n{model} Recommendations:")
        for _, row in best_by_gpu.iterrows():
            print(f"  {row['gpu']}: Use {row['precision']} precision "
                  f"({row['throughput_samples_per_sec']:.0f} samples/sec)")
        
        # Memory efficiency analysis
        memory_efficient = model_data.nsmallest(1, 'peak_memory_mb')
        performance_leader = model_data.nlargest(1, 'throughput_samples_per_sec')
        
        print(f"  Most Memory Efficient: {memory_efficient.iloc[0]['precision']} "
              f"({memory_efficient.iloc[0]['peak_memory_mb']:.0f} MB)")
        print(f"  Highest Performance: {performance_leader.iloc[0]['precision']} "
              f"({performance_leader.iloc[0]['throughput_samples_per_sec']:.0f} samples/sec)")

else:
    print("No inference data available for model-specific analysis")

No inference data available for model-specific analysis


## 5. Precision Format Stability Analysis

This section analyzes the numerical stability and convergence behavior of different precision formats, including gradient analysis and loss curve comparisons.

In [13]:
def analyze_precision_stability(inference_df, training_df=None):
    """Analyze numerical stability across precision formats"""
    
    stability_analysis = {}
    
    # Inference stability analysis
    if not inference_df.empty:
        print("=== INFERENCE STABILITY ANALYSIS ===")
        
        # Coefficient of variation (CV) analysis
        inference_stability = inference_df.groupby(['model', 'precision']).agg({
            'throughput_samples_per_sec': ['mean', 'std', 'count'],
            'mean_batch_latency_ms': ['mean', 'std'],
            'peak_memory_mb': ['mean', 'std']
        })
        
        # Calculate coefficient of variation
        inference_stability['throughput_cv'] = (
            inference_stability[('throughput_samples_per_sec', 'std')] / 
            inference_stability[('throughput_samples_per_sec', 'mean')]
        )
        
        inference_stability['latency_cv'] = (
            inference_stability[('mean_batch_latency_ms', 'std')] / 
            inference_stability[('mean_batch_latency_ms', 'mean')]
        )
        
        stability_analysis['inference'] = inference_stability
        
        print("Throughput Coefficient of Variation by Model and Precision:")
        cv_summary = inference_df.groupby(['model', 'precision']).agg({
            'throughput_samples_per_sec': lambda x: x.std() / x.mean()
        }).round(4)
        print(cv_summary)
    
    # Training stability analysis (if available)
    if training_df is not None and not training_df.empty:
        print("\n=== TRAINING STABILITY ANALYSIS ===")
        
        # Analyze loss convergence if available
        if 'loss' in training_df.columns:
            training_stability = training_df.groupby(['model', 'precision']).agg({
                'loss': ['mean', 'std', 'min', 'max'],
                'samples_per_sec': ['mean', 'std'] if 'samples_per_sec' in training_df.columns else ['count']
            })
            
            stability_analysis['training'] = training_stability
            
            print("Training Loss Statistics by Model and Precision:")
            print(training_stability)
    
    return stability_analysis

def plot_stability_analysis(df):
    """Visualize precision format stability"""
    
    # Calculate variance metrics
    variance_data = []
    
    for (model, precision), group in df.groupby(['model', 'precision']):
        if len(group) > 1:  # Need multiple samples for variance
            throughput_cv = group['throughput_samples_per_sec'].std() / group['throughput_samples_per_sec'].mean()
            latency_cv = group['mean_batch_latency_ms'].std() / group['mean_batch_latency_ms'].mean()
            memory_cv = group['peak_memory_mb'].std() / group['peak_memory_mb'].mean()
            
            variance_data.append({
                'model': model,
                'precision': precision,
                'throughput_cv': throughput_cv,
                'latency_cv': latency_cv,
                'memory_cv': memory_cv
            })
    
    if not variance_data:
        print("Insufficient data for stability analysis")
        return
    
    variance_df = pd.DataFrame(variance_data)
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Throughput stability
    sns.barplot(data=variance_df, x='precision', y='throughput_cv', hue='model', ax=axes[0,0])
    axes[0,0].set_title('Throughput Stability (Lower CV = More Stable)')
    axes[0,0].set_ylabel('Coefficient of Variation')
    
    # Latency stability
    sns.barplot(data=variance_df, x='precision', y='latency_cv', hue='model', ax=axes[0,1])
    axes[0,1].set_title('Latency Stability (Lower CV = More Stable)')
    axes[0,1].set_ylabel('Coefficient of Variation')
    
    # Memory stability
    sns.barplot(data=variance_df, x='precision', y='memory_cv', hue='model', ax=axes[1,0])
    axes[1,0].set_title('Memory Usage Stability (Lower CV = More Stable)')
    axes[1,0].set_ylabel('Coefficient of Variation')
    
    # Stability heatmap
    stability_matrix = variance_df.pivot_table(
        values='throughput_cv', index='model', columns='precision', aggfunc='mean'
    )
    sns.heatmap(stability_matrix, annot=True, fmt='.4f', cmap='RdYlBu_r', ax=axes[1,1])
    axes[1,1].set_title('Throughput Stability Heatmap')
    
    plt.tight_layout()
    plt.show()
    
    return variance_df

def analyze_numerical_ranges(df):
    """Analyze numerical ranges and potential precision issues"""
    
    print("\n=== NUMERICAL RANGE ANALYSIS ===")
    
    # Analyze metric ranges by precision
    range_analysis = df.groupby('precision').agg({
        'throughput_samples_per_sec': ['min', 'max', 'mean'],
        'mean_batch_latency_ms': ['min', 'max', 'mean'],
        'peak_memory_mb': ['min', 'max', 'mean']
    }).round(3)
    
    print("Metric Ranges by Precision Format:")
    print(range_analysis)
    
    # Check for outliers (values beyond 3 standard deviations)
    outlier_analysis = {}
    
    for precision in df['precision'].unique():
        precision_data = df[df['precision'] == precision]
        
        outliers = {}
        for metric in ['throughput_samples_per_sec', 'mean_batch_latency_ms', 'peak_memory_mb']:
            mean = precision_data[metric].mean()
            std = precision_data[metric].std()
            
            # Identify outliers
            outlier_mask = np.abs(precision_data[metric] - mean) > 3 * std
            outliers[metric] = outlier_mask.sum()
        
        outlier_analysis[precision] = outliers
    
    print("\nOutlier Analysis (values beyond 3σ):")
    outlier_df = pd.DataFrame(outlier_analysis).T
    print(outlier_df)
    
    return range_analysis, outlier_df

def precision_robustness_test(df):
    """Test robustness of precision formats across different conditions"""
    
    print("\n=== PRECISION ROBUSTNESS TEST ===")
    
    robustness_scores = {}
    
    for precision in df['precision'].unique():
        precision_data = df[df['precision'] == precision]
        
        # Calculate robustness metrics
        # 1. Consistency across different GPUs
        gpu_variance = precision_data.groupby('gpu')['throughput_samples_per_sec'].std().mean()
        
        # 2. Consistency across different models
        model_variance = precision_data.groupby('model')['throughput_samples_per_sec'].std().mean()
        
        # 3. Overall coefficient of variation
        overall_cv = precision_data['throughput_samples_per_sec'].std() / precision_data['throughput_samples_per_sec'].mean()
        
        # 4. Performance spread (max - min)
        performance_spread = (precision_data['throughput_samples_per_sec'].max() - 
                            precision_data['throughput_samples_per_sec'].min())
        
        robustness_scores[precision] = {
            'gpu_variance': gpu_variance,
            'model_variance': model_variance,
            'overall_cv': overall_cv,
            'performance_spread': performance_spread,
            'robustness_score': 1 / (1 + overall_cv)  # Higher score = more robust
        }
    
    robustness_df = pd.DataFrame(robustness_scores).T
    print("Precision Format Robustness Analysis:")
    print(robustness_df.round(4))
    
    # Rank by robustness
    robustness_ranking = robustness_df.sort_values('robustness_score', ascending=False)
    print("\nRobustness Ranking (Higher = More Robust):")
    for i, (precision, score) in enumerate(robustness_ranking['robustness_score'].items(), 1):
        print(f"{i}. {precision}: {score:.4f}")
    
    return robustness_df

# Perform stability analysis
if not inference_df.empty:
    print("=== PRECISION FORMAT STABILITY ANALYSIS ===")
    
    # Main stability analysis
    stability_results = analyze_precision_stability(inference_df, training_df)
    
    # Create stability visualizations
    print("\nCreating stability visualizations...")
    variance_results = plot_stability_analysis(inference_df)
    
    # Numerical range analysis
    range_results, outlier_results = analyze_numerical_ranges(inference_df)
    
    # Robustness testing
    robustness_results = precision_robustness_test(inference_df)
    
    # Summary insights
    print("\n=== STABILITY INSIGHTS SUMMARY ===")
    
    if variance_results is not None and not variance_results.empty:
        # Find most stable precision
        avg_stability = variance_results.groupby('precision')[['throughput_cv', 'latency_cv', 'memory_cv']].mean()
        most_stable = avg_stability.sum(axis=1).idxmin()
        least_stable = avg_stability.sum(axis=1).idxmax()
        
        print(f"Most Stable Precision Format: {most_stable}")
        print(f"Least Stable Precision Format: {least_stable}")
        
        print("\nStability Metrics (Lower = Better):")
        print(avg_stability.round(4))
    
    # Precision-specific recommendations
    print("\n=== PRECISION STABILITY RECOMMENDATIONS ===")
    
    for precision in inference_df['precision'].unique():
        precision_data = inference_df[inference_df['precision'] == precision]
        
        # Calculate stability metrics
        throughput_cv = precision_data['throughput_samples_per_sec'].std() / precision_data['throughput_samples_per_sec'].mean()
        
        stability_rating = "High" if throughput_cv < 0.1 else "Medium" if throughput_cv < 0.2 else "Low"
        
        print(f"\n{precision}:")
        print(f"  Stability Rating: {stability_rating} (CV: {throughput_cv:.4f})")
        print(f"  Recommended for: {'Production use' if throughput_cv < 0.15 else 'Development/testing'}")
        
        # Best use cases
        best_models = precision_data.groupby('model')['throughput_samples_per_sec'].mean().nlargest(2)
        print(f"  Best performance with: {', '.join(best_models.index)}")

else:
    print("No inference data available for stability analysis")

No inference data available for stability analysis


## 6. Hardware Utilization Metrics

This section calculates and visualizes hardware utilization efficiency, including compute unit usage, memory bandwidth utilization, and bottleneck identification.

In [9]:
def analyze_hardware_utilization(df):
    """Analyze hardware utilization efficiency across configurations"""
    
    if 'gpu_avg_util_percent' not in df.columns:
        print("GPU utilization data not available")
        return None
    
    # Hardware utilization analysis
    hw_analysis = df.groupby(['gpu', 'model', 'precision']).agg({
        'gpu_avg_util_percent': ['mean', 'std'],
        'throughput_samples_per_sec': 'mean',
        'peak_memory_mb': 'mean',
        'gpu_avg_power_watts': 'mean' if 'gpu_avg_power_watts' in df.columns else lambda x: 0
    }).round(2)
    
    hw_analysis.columns = ['_'.join(col).strip() for col in hw_analysis.columns]
    hw_analysis = hw_analysis.reset_index()
    
    # Calculate utilization efficiency (throughput per utilization percent)
    hw_analysis['utilization_efficiency'] = (
        hw_analysis['throughput_samples_per_sec_mean'] / 
        hw_analysis['gpu_avg_util_percent_mean']
    )
    
    # Identify bottlenecks
    hw_analysis['utilization_category'] = hw_analysis['gpu_avg_util_percent_mean'].apply(
        lambda x: 'High' if x >= 80 else 'Medium' if x >= 60 else 'Low'
    )
    
    return hw_analysis

def plot_hardware_utilization(hw_df, inference_df):
    """Create hardware utilization visualizations"""
    
    if hw_df is None:
        print("No hardware utilization data to plot")
        return
    
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    
    # 1. GPU Utilization by configuration
    sns.barplot(data=hw_df, x='precision', y='gpu_avg_util_percent_mean', 
                hue='model', ax=axes[0,0])
    axes[0,0].set_title('GPU Utilization by Precision')
    axes[0,0].set_ylabel('GPU Utilization (%)')
    axes[0,0].legend(title='Model')
    
    # 2. Utilization efficiency
    sns.barplot(data=hw_df, x='gpu', y='utilization_efficiency', 
                hue='precision', ax=axes[0,1])
    axes[0,1].set_title('Utilization Efficiency by GPU')
    axes[0,1].set_ylabel('Throughput per Utilization %')
    axes[0,1].tick_params(axis='x', rotation=45)
    axes[0,1].legend(title='Precision')
    
    # 3. Utilization vs Performance scatter
    sns.scatterplot(data=hw_df, x='gpu_avg_util_percent_mean', y='throughput_samples_per_sec_mean',
                   hue='precision', style='model', s=150, ax=axes[0,2])
    axes[0,2].set_title('GPU Utilization vs Performance')
    axes[0,2].set_xlabel('GPU Utilization (%)')
    axes[0,2].set_ylabel('Throughput (samples/sec)')
    
    # 4. Utilization distribution
    sns.boxplot(data=inference_df, x='precision', y='gpu_avg_util_percent', ax=axes[1,0])
    axes[1,0].set_title('GPU Utilization Distribution')
    axes[1,0].set_ylabel('GPU Utilization (%)')
    
    # 5. Utilization heatmap by GPU and Model
    util_pivot = hw_df.pivot_table(
        values='gpu_avg_util_percent_mean', 
        index='model', 
        columns='gpu', 
        aggfunc='mean'
    )
    sns.heatmap(util_pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[1,1])
    axes[1,1].set_title('GPU Utilization Heatmap (%)')
    
    # 6. Bottleneck analysis
    bottleneck_counts = hw_df['utilization_category'].value_counts()
    axes[1,2].pie(bottleneck_counts.values, labels=bottleneck_counts.index, autopct='%1.1f%%')
    axes[1,2].set_title('GPU Utilization Distribution')
    
    plt.tight_layout()
    plt.show()

def identify_performance_bottlenecks(df):
    """Identify potential performance bottlenecks"""
    
    bottlenecks = {}
    
    for (gpu, model, precision), group in df.groupby(['gpu', 'model', 'precision']):
        config_name = f"{gpu}_{model}_{precision}"
        
        avg_util = group['gpu_avg_util_percent'].mean() if 'gpu_avg_util_percent' in group.columns else 0
        avg_memory = group['peak_memory_mb'].mean()
        avg_throughput = group['throughput_samples_per_sec'].mean()
        
        # Bottleneck analysis
        bottleneck_type = []
        
        if avg_util < 70:
            bottleneck_type.append("GPU_UNDERUTILIZED")
        elif avg_util > 95:
            bottleneck_type.append("GPU_SATURATED")
        
        # Memory bottleneck (relative to known GPU memory limits)
        gpu_memory_limits = {
            'NVIDIA_GeForce_RTX_4090': 24000,
            'NVIDIA_L40S': 48000,
            'Radeon_RX_7900_XT': 20000
        }
        
        if gpu in gpu_memory_limits:
            memory_usage_percent = (avg_memory / gpu_memory_limits[gpu]) * 100
            if memory_usage_percent > 80:
                bottleneck_type.append("MEMORY_LIMITED")
        
        # Performance bottleneck (low throughput relative to utilization)
        if avg_util > 80 and avg_throughput < 500:  # Threshold depends on use case
            bottleneck_type.append("COMPUTE_BOUND")
        
        bottlenecks[config_name] = {
            'gpu': gpu,
            'model': model,
            'precision': precision,
            'avg_utilization': avg_util,
            'avg_memory_mb': avg_memory,
            'avg_throughput': avg_throughput,
            'bottlenecks': bottleneck_type if bottleneck_type else ["NONE"]
        }
    
    return bottlenecks

def calculate_theoretical_performance(df):
    """Calculate theoretical vs actual performance ratios"""
    
    # Theoretical peak performance (simplified estimates)
    gpu_theoretical_specs = {
        'NVIDIA_GeForce_RTX_4090': {'fp32_tflops': 83, 'fp16_tflops': 165, 'memory_bandwidth_gbps': 1008},
        'NVIDIA_L40S': {'fp32_tflops': 91, 'fp16_tflops': 183, 'memory_bandwidth_gbps': 864},
        'Radeon_RX_7900_XT': {'fp32_tflops': 61, 'fp16_tflops': 122, 'memory_bandwidth_gbps': 960}
    }
    
    performance_ratios = []
    
    for (gpu, precision), group in df.groupby(['gpu', 'precision']):
        if gpu in gpu_theoretical_specs:
            specs = gpu_theoretical_specs[gpu]
            
            # Estimate theoretical performance based on precision
            if 'fp32' in precision:
                theoretical_tflops = specs['fp32_tflops']
            else:
                theoretical_tflops = specs['fp16_tflops']
            
            avg_utilization = group['gpu_avg_util_percent'].mean() if 'gpu_avg_util_percent' in group.columns else 100
            avg_throughput = group['throughput_samples_per_sec'].mean()
            
            # Calculate efficiency ratio (simplified)
            utilization_ratio = avg_utilization / 100
            
            performance_ratios.append({
                'gpu': gpu,
                'precision': precision,
                'theoretical_tflops': theoretical_tflops,
                'utilization_ratio': utilization_ratio,
                'avg_throughput': avg_throughput,
                'memory_bandwidth_gbps': specs['memory_bandwidth_gbps']
            })
    
    return pd.DataFrame(performance_ratios)

# Perform hardware utilization analysis
if not inference_df.empty:
    print("=== HARDWARE UTILIZATION ANALYSIS ===")
    
    hw_util_results = analyze_hardware_utilization(inference_df)
    
    if hw_util_results is not None:
        print("\nHardware Utilization Summary:")
        display_cols = ['gpu', 'model', 'precision', 'gpu_avg_util_percent_mean', 
                       'utilization_efficiency', 'utilization_category']
        print(hw_util_results[display_cols].to_string(index=False))
        
        # Utilization insights
        print("\n=== UTILIZATION INSIGHTS ===")
        
        # Best utilization efficiency
        best_efficiency = hw_util_results.nlargest(3, 'utilization_efficiency')
        print("\nMost Efficient Utilization:")
        for _, row in best_efficiency.iterrows():
            print(f"  {row['gpu']} + {row['model']} + {row['precision']}: "
                  f"{row['utilization_efficiency']:.2f} samples/sec/%util")
        
        # Utilization distribution
        util_dist = hw_util_results['utilization_category'].value_counts()
        print(f"\nUtilization Distribution:")
        for category, count in util_dist.items():
            print(f"  {category} utilization: {count} configurations")
        
        # Create visualizations
        print("\nCreating hardware utilization visualizations...")
        plot_hardware_utilization(hw_util_results, inference_df)
        
        # Bottleneck analysis
        print("\n=== BOTTLENECK ANALYSIS ===")
        bottlenecks = identify_performance_bottlenecks(inference_df)
        
        print("Performance Bottleneck Analysis:")
        for config, analysis in bottlenecks.items():
            if "NONE" not in analysis['bottlenecks']:
                print(f"\n{config}:")
                print(f"  GPU: {analysis['gpu']}")
                print(f"  Utilization: {analysis['avg_utilization']:.1f}%")
                print(f"  Memory: {analysis['avg_memory_mb']:.0f} MB")
                print(f"  Throughput: {analysis['avg_throughput']:.0f} samples/sec")
                print(f"  Bottlenecks: {', '.join(analysis['bottlenecks'])}")
        
        # Theoretical performance comparison
        print("\n=== THEORETICAL VS ACTUAL PERFORMANCE ===")
        theoretical_perf = calculate_theoretical_performance(inference_df)
        
        if not theoretical_perf.empty:
            print("GPU Efficiency Analysis:")
            print(theoretical_perf.to_string(index=False))
        
        # Hardware recommendations
        print("\n=== HARDWARE UTILIZATION RECOMMENDATIONS ===")
        
        for gpu in inference_df['gpu'].unique():
            gpu_data = hw_util_results[hw_util_results['gpu'] == gpu]
            
            # Find best configurations for this GPU
            best_config = gpu_data.loc[gpu_data['utilization_efficiency'].idxmax()]
            avg_util = gpu_data['gpu_avg_util_percent_mean'].mean()
            
            print(f"\n{gpu}:")
            print(f"  Average Utilization: {avg_util:.1f}%")
            print(f"  Best Configuration: {best_config['model']} + {best_config['precision']}")
            print(f"  Peak Efficiency: {best_config['utilization_efficiency']:.2f} samples/sec/%util")
            
            # Recommendations
            if avg_util < 60:
                print("  Recommendation: Increase batch size or optimize data pipeline")
            elif avg_util > 90:
                print("  Recommendation: Consider memory optimizations or reduce batch size")
            else:
                print("  Recommendation: Well-balanced utilization")

else:
    print("No inference data available for hardware utilization analysis")

No inference data available for hardware utilization analysis


## 7. Batch Size Optimization

This section implements optimization algorithms to find optimal batch sizes for each hardware-precision combination, balancing throughput and memory constraints.

In [14]:
def optimize_batch_sizes(df):
    """Find optimal batch sizes for each configuration"""
    
    optimization_results = {}
    
    # Define GPU memory limits (MB)
    gpu_memory_limits = {
        'NVIDIA_GeForce_RTX_4090': 24000,
        'NVIDIA_L40S': 48000,
        'Radeon_RX_7900_XT': 20000
    }
    
    for (gpu, model, precision), group in df.groupby(['gpu', 'model', 'precision']):
        
        if len(group) < 2:
            continue
            
        # Sort by batch size
        group_sorted = group.sort_values('batch_size')
        
        # Performance vs batch size analysis
        batch_sizes = group_sorted['batch_size'].values
        throughputs = group_sorted['throughput_samples_per_sec'].values
        memory_usage = group_sorted['peak_memory_mb'].values
        
        # Find optimal batch size using different criteria
        
        # 1. Maximum throughput
        max_throughput_idx = np.argmax(throughputs)
        optimal_max_throughput = {
            'batch_size': batch_sizes[max_throughput_idx],
            'throughput': throughputs[max_throughput_idx],
            'memory': memory_usage[max_throughput_idx]
        }
        
        # 2. Best throughput per memory ratio
        throughput_per_mb = throughputs / memory_usage
        best_efficiency_idx = np.argmax(throughput_per_mb)
        optimal_efficiency = {
            'batch_size': batch_sizes[best_efficiency_idx],
            'throughput': throughputs[best_efficiency_idx],
            'memory': memory_usage[best_efficiency_idx],
            'efficiency': throughput_per_mb[best_efficiency_idx]
        }
        
        # 3. Memory constraint optimization (80% of GPU memory)
        memory_limit = gpu_memory_limits.get(gpu, 20000) * 0.8
        valid_memory_mask = memory_usage <= memory_limit
        
        if np.any(valid_memory_mask):
            valid_throughputs = throughputs[valid_memory_mask]
            valid_batch_sizes = batch_sizes[valid_memory_mask]
            valid_memory = memory_usage[valid_memory_mask]
            
            best_valid_idx = np.argmax(valid_throughputs)
            optimal_memory_constrained = {
                'batch_size': valid_batch_sizes[best_valid_idx],
                'throughput': valid_throughputs[best_valid_idx],
                'memory': valid_memory[best_valid_idx]
            }
        else:
            optimal_memory_constrained = optimal_efficiency
        
        # 4. Pareto optimal points (throughput vs memory trade-off)
        pareto_optimal = []
        for i, (bs, tp, mem) in enumerate(zip(batch_sizes, throughputs, memory_usage)):
            is_pareto = True
            for j, (bs2, tp2, mem2) in enumerate(zip(batch_sizes, throughputs, memory_usage)):
                if i != j and tp2 >= tp and mem2 <= mem and (tp2 > tp or mem2 < mem):
                    is_pareto = False
                    break
            if is_pareto:
                pareto_optimal.append({'batch_size': bs, 'throughput': tp, 'memory': mem})
        
        optimization_results[f"{gpu}_{model}_{precision}"] = {
            'gpu': gpu,
            'model': model,
            'precision': precision,
            'optimal_max_throughput': optimal_max_throughput,
            'optimal_efficiency': optimal_efficiency,
            'optimal_memory_constrained': optimal_memory_constrained,
            'pareto_optimal': pareto_optimal,
            'all_data': {
                'batch_sizes': batch_sizes.tolist(),
                'throughputs': throughputs.tolist(),
                'memory_usage': memory_usage.tolist()
            }
        }
    
    return optimization_results

def plot_batch_size_optimization(optimization_results):
    """Visualize batch size optimization results"""
    
    n_configs = len(optimization_results)
    n_cols = min(3, n_configs)
    n_rows = (n_configs + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 5*n_rows))
    if n_configs == 1:
        axes = [axes]
    elif n_rows == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    
    for i, (config_name, results) in enumerate(optimization_results.items()):
        if i >= len(axes):
            break
            
        ax = axes[i]
        
        # Plot throughput vs batch size
        batch_sizes = results['all_data']['batch_sizes']
        throughputs = results['all_data']['throughputs']
        memory_usage = results['all_data']['memory_usage']
        
        # Throughput curve
        ax.plot(batch_sizes, throughputs, 'b-o', label='Throughput', markersize=6)
        
        # Mark optimal points
        optimal_max = results['optimal_max_throughput']
        optimal_eff = results['optimal_efficiency']
        optimal_mem = results['optimal_memory_constrained']
        
        ax.scatter([optimal_max['batch_size']], [optimal_max['throughput']], 
                  color='red', s=100, marker='*', label='Max Throughput', zorder=5)
        ax.scatter([optimal_eff['batch_size']], [optimal_eff['throughput']], 
                  color='green', s=100, marker='s', label='Best Efficiency', zorder=5)
        ax.scatter([optimal_mem['batch_size']], [optimal_mem['throughput']], 
                  color='orange', s=100, marker='^', label='Memory Constrained', zorder=5)
        
        # Pareto frontier
        pareto_points = results['pareto_optimal']
        if pareto_points:
            pareto_bs = [p['batch_size'] for p in pareto_points]
            pareto_tp = [p['throughput'] for p in pareto_points]
            ax.plot(pareto_bs, pareto_tp, 'r--', alpha=0.7, label='Pareto Frontier')
        
        ax.set_title(f"{results['gpu']}\n{results['model']} + {results['precision']}")
        ax.set_xlabel('Batch Size')
        ax.set_ylabel('Throughput (samples/sec)')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    
    # Hide unused subplots
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    
    plt.tight_layout()
    plt.show()

def create_optimization_summary(optimization_results):
    """Create summary table of optimization results"""
    
    summary_data = []
    
    for config_name, results in optimization_results.items():
        
        # Extract optimal configurations
        max_tp = results['optimal_max_throughput']
        best_eff = results['optimal_efficiency']
        mem_const = results['optimal_memory_constrained']
        
        summary_data.append({
            'gpu': results['gpu'],
            'model': results['model'],
            'precision': results['precision'],
            'max_throughput_bs': max_tp['batch_size'],
            'max_throughput_value': max_tp['throughput'],
            'max_throughput_memory': max_tp['memory'],
            'best_efficiency_bs': best_eff['batch_size'],
            'best_efficiency_value': best_eff['throughput'],
            'best_efficiency_memory': best_eff['memory'],
            'memory_constrained_bs': mem_const['batch_size'],
            'memory_constrained_value': mem_const['throughput'],
            'memory_constrained_memory': mem_const['memory']
        })
    
    return pd.DataFrame(summary_data)

def recommend_batch_sizes(optimization_results):
    """Provide batch size recommendations based on use case"""
    
    recommendations = {}
    
    for config_name, results in optimization_results.items():
        gpu = results['gpu']
        model = results['model']
        precision = results['precision']
        
        max_tp = results['optimal_max_throughput']
        best_eff = results['optimal_efficiency']
        mem_const = results['optimal_memory_constrained']
        
        # Use case recommendations
        use_cases = {
            'high_throughput': {
                'description': 'Maximum inference speed (production)',
                'batch_size': max_tp['batch_size'],
                'expected_throughput': max_tp['throughput'],
                'memory_usage': max_tp['memory']
            },
            'memory_efficient': {
                'description': 'Best performance per MB',
                'batch_size': best_eff['batch_size'],
                'expected_throughput': best_eff['throughput'],
                'memory_usage': best_eff['memory']
            },
            'memory_safe': {
                'description': 'Conservative memory usage',
                'batch_size': mem_const['batch_size'],
                'expected_throughput': mem_const['throughput'],
                'memory_usage': mem_const['memory']
            }
        }
        
        recommendations[config_name] = {
            'gpu': gpu,
            'model': model,
            'precision': precision,
            'use_cases': use_cases
        }
    
    return recommendations

# Perform batch size optimization
if not inference_df.empty:
    print("=== BATCH SIZE OPTIMIZATION ANALYSIS ===")
    
    # Check if we have sufficient data for optimization
    batch_size_counts = inference_df.groupby(['gpu', 'model', 'precision']).size()
    configs_with_multiple_bs = batch_size_counts[batch_size_counts >= 2]
    
    if len(configs_with_multiple_bs) == 0:
        print("Insufficient batch size variations for optimization analysis")
        print("Available data points per configuration:")
        print(batch_size_counts)
    else:
        print(f"Analyzing {len(configs_with_multiple_bs)} configurations with multiple batch sizes")
        
        # Perform optimization
        optimization_results = optimize_batch_sizes(inference_df)
        
        if optimization_results:
            print("\n=== OPTIMIZATION RESULTS ===")
            
            # Create summary table
            summary_df = create_optimization_summary(optimization_results)
            print("\nOptimal Batch Size Summary:")
            print(summary_df.to_string(index=False))
            
            # Create visualizations
            print("\nCreating batch size optimization visualizations...")
            plot_batch_size_optimization(optimization_results)
            
            # Generate recommendations
            print("\n=== BATCH SIZE RECOMMENDATIONS ===")
            recommendations = recommend_batch_sizes(optimization_results)
            
            for config_name, rec in recommendations.items():
                print(f"\n--- {rec['gpu']} + {rec['model']} + {rec['precision']} ---")
                
                for use_case, details in rec['use_cases'].items():
                    print(f"\n{use_case.replace('_', ' ').title()}:")
                    print(f"  Batch Size: {details['batch_size']}")
                    print(f"  Expected Throughput: {details['expected_throughput']:.0f} samples/sec")
                    print(f"  Memory Usage: {details['memory_usage']:.0f} MB")
                    print(f"  Use Case: {details['description']}")
            
            # Overall insights
            print("\n=== OPTIMIZATION INSIGHTS ===")
            
            # Find configurations with biggest batch size impact
            batch_impact = []
            for config_name, results in optimization_results.items():
                throughputs = results['all_data']['throughputs']
                if len(throughputs) > 1:
                    impact = (max(throughputs) - min(throughputs)) / min(throughputs) * 100
                    batch_impact.append({
                        'config': config_name,
                        'impact_percent': impact
                    })
            
            if batch_impact:
                batch_impact_df = pd.DataFrame(batch_impact).sort_values('impact_percent', ascending=False)
                print("\nConfigurations with highest batch size sensitivity:")
                for _, row in batch_impact_df.head(5).iterrows():
                    print(f"  {row['config']}: {row['impact_percent']:.1f}% performance difference")
            
            # Memory efficiency analysis
            print("\n=== MEMORY EFFICIENCY ANALYSIS ===")
            
            for config_name, results in optimization_results.items():
                batch_sizes = results['all_data']['batch_sizes']
                throughputs = results['all_data']['throughputs']
                memory_usage = results['all_data']['memory_usage']
                
                # Calculate throughput per MB for each batch size
                efficiency_ratios = np.array(throughputs) / np.array(memory_usage)
                best_efficiency_idx = np.argmax(efficiency_ratios)
                
                print(f"\n{config_name}:")
                print(f"  Most efficient batch size: {batch_sizes[best_efficiency_idx]} "
                      f"({efficiency_ratios[best_efficiency_idx]:.3f} samples/sec/MB)")
                print(f"  Memory scaling factor: {memory_usage[-1]/memory_usage[0]:.2f}x "
                      f"(batch size {batch_sizes[0]} → {batch_sizes[-1]})")
        
else:
    print("No inference data available for batch size optimization")

No inference data available for batch size optimization


## 8. Performance Regression Analysis

This section builds regression models to predict performance metrics based on hardware, model, and precision parameters, enabling performance forecasting for new configurations.

In [11]:
def build_performance_regression_model(df):
    """Build regression models to predict performance metrics"""
    
    if df.empty:
        print("No data available for regression analysis")
        return None
    
    # Prepare features for regression
    model_data = df.copy()
    
    # Encode categorical variables
    le_gpu = LabelEncoder()
    le_model = LabelEncoder()
    le_precision = LabelEncoder()
    
    model_data['gpu_encoded'] = le_gpu.fit_transform(model_data['gpu'])
    model_data['model_encoded'] = le_model.fit_transform(model_data['model'])
    model_data['precision_encoded'] = le_precision.fit_transform(model_data['precision'])
    
    # Feature matrix
    features = ['gpu_encoded', 'model_encoded', 'precision_encoded', 'batch_size']
    X = model_data[features]
    
    # Target variables
    targets = {
        'throughput': 'throughput_samples_per_sec',
        'latency': 'mean_batch_latency_ms',
        'memory': 'peak_memory_mb'
    }
    
    models = {}
    results = {}
    
    for target_name, target_col in targets.items():
        if target_col in model_data.columns:
            y = model_data[target_col].dropna()
            X_clean = X.loc[y.index]
            
            # Split data
            X_train, X_test, y_train, y_test = train_test_split(
                X_clean, y, test_size=0.2, random_state=42
            )
            
            # Scale features
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            
            # Train model
            model = LinearRegression()
            model.fit(X_train_scaled, y_train)
            
            # Predictions
            y_pred_train = model.predict(X_train_scaled)
            y_pred_test = model.predict(X_test_scaled)
            
            # Evaluation metrics
            train_r2 = r2_score(y_train, y_pred_train)
            test_r2 = r2_score(y_test, y_pred_test)
            train_mae = mean_absolute_error(y_train, y_pred_train)
            test_mae = mean_absolute_error(y_test, y_pred_test)
            
            models[target_name] = {
                'model': model,
                'scaler': scaler,
                'encoders': {'gpu': le_gpu, 'model': le_model, 'precision': le_precision},
                'feature_names': features,
                'target_column': target_col
            }
            
            results[target_name] = {
                'train_r2': train_r2,
                'test_r2': test_r2,
                'train_mae': train_mae,
                'test_mae': test_mae,
                'feature_importance': dict(zip(features, model.coef_)),
                'intercept': model.intercept_,
                'predictions': {
                    'y_test': y_test.values,
                    'y_pred_test': y_pred_test,
                    'y_train': y_train.values,
                    'y_pred_train': y_pred_train
                }
            }
    
    return models, results

def plot_regression_analysis(results):
    """Visualize regression model performance"""
    
    n_targets = len(results)
    fig, axes = plt.subplots(2, n_targets, figsize=(6*n_targets, 12))
    
    if n_targets == 1:
        axes = axes.reshape(-1, 1)
    
    target_names = list(results.keys())
    
    for i, (target_name, result) in enumerate(results.items()):
        
        # Actual vs Predicted (Test Set)
        y_test = result['predictions']['y_test']
        y_pred_test = result['predictions']['y_pred_test']
        
        axes[0, i].scatter(y_test, y_pred_test, alpha=0.6)
        axes[0, i].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
        axes[0, i].set_xlabel(f'Actual {target_name}')
        axes[0, i].set_ylabel(f'Predicted {target_name}')
        axes[0, i].set_title(f'{target_name.title()} Prediction\nR² = {result["test_r2"]:.3f}')
        axes[0, i].grid(True, alpha=0.3)
        
        # Feature importance
        importance = result['feature_importance']
        features = list(importance.keys())
        values = list(importance.values())
        
        bars = axes[1, i].bar(features, values)
        axes[1, i].set_title(f'{target_name.title()} Feature Importance')
        axes[1, i].set_ylabel('Coefficient Value')
        axes[1, i].tick_params(axis='x', rotation=45)
        
        # Color bars by importance
        for bar, val in zip(bars, values):
            if val > 0:
                bar.set_color('green')
            else:
                bar.set_color('red')
        
        axes[1, i].axhline(y=0, color='black', linestyle='-', alpha=0.3)
        axes[1, i].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def predict_performance(models, gpu, model_arch, precision, batch_size):
    """Predict performance for a given configuration"""
    
    predictions = {}
    
    for target_name, model_info in models.items():
        try:
            # Encode inputs
            gpu_encoded = model_info['encoders']['gpu'].transform([gpu])[0]
            model_encoded = model_info['encoders']['model'].transform([model_arch])[0]
            precision_encoded = model_info['encoders']['precision'].transform([precision])[0]
            
            # Create feature vector
            features = np.array([[gpu_encoded, model_encoded, precision_encoded, batch_size]])
            
            # Scale features
            features_scaled = model_info['scaler'].transform(features)
            
            # Make prediction
            prediction = model_info['model'].predict(features_scaled)[0]
            predictions[target_name] = prediction
            
        except ValueError as e:
            predictions[target_name] = f"Error: {e}"
    
    return predictions

def analyze_performance_factors(results):
    """Analyze which factors most influence performance"""
    
    factor_analysis = {}
    
    for target_name, result in results.items():
        importance = result['feature_importance']
        
        # Normalize importance values
        max_abs_importance = max(abs(val) for val in importance.values())
        normalized_importance = {
            feature: abs(val) / max_abs_importance 
            for feature, val in importance.items()
        }
        
        # Rank factors
        ranked_factors = sorted(normalized_importance.items(), key=lambda x: x[1], reverse=True)
        
        factor_analysis[target_name] = {
            'raw_importance': importance,
            'normalized_importance': normalized_importance,
            'ranked_factors': ranked_factors,
            'r2_score': result['test_r2']
        }
    
    return factor_analysis

def create_performance_predictions(models, df):
    """Create predictions for all possible configurations"""
    
    if not models:
        return None
    
    # Get unique values for each categorical variable
    unique_gpus = df['gpu'].unique()
    unique_models = df['model'].unique()
    unique_precisions = df['precision'].unique()
    batch_sizes = [16, 32, 64, 128, 256]  # Standard batch sizes
    
    predictions = []
    
    for gpu in unique_gpus:
        for model in unique_models:
            for precision in unique_precisions:
                for batch_size in batch_sizes:
                    
                    pred = predict_performance(models, gpu, model, precision, batch_size)
                    
                    if all(isinstance(val, (int, float)) for val in pred.values()):
                        predictions.append({
                            'gpu': gpu,
                            'model': model,
                            'precision': precision,
                            'batch_size': batch_size,
                            **pred
                        })
    
    return pd.DataFrame(predictions) if predictions else None

# Perform regression analysis
if not inference_df.empty:
    print("=== PERFORMANCE REGRESSION ANALYSIS ===")
    
    # Build regression models
    regression_models, regression_results = build_performance_regression_model(inference_df)
    
    if regression_models and regression_results:
        print("\nRegression Model Performance:")
        for target_name, result in regression_results.items():
            print(f"\n{target_name.upper()} Model:")
            print(f"  Training R²: {result['train_r2']:.4f}")
            print(f"  Test R²: {result['test_r2']:.4f}")
            print(f"  Training MAE: {result['train_mae']:.2f}")
            print(f"  Test MAE: {result['test_mae']:.2f}")
        
        # Create visualizations
        print("\nCreating regression analysis visualizations...")
        plot_regression_analysis(regression_results)
        
        # Factor analysis
        print("\n=== PERFORMANCE FACTOR ANALYSIS ===")
        factor_analysis = analyze_performance_factors(regression_results)
        
        for target_name, analysis in factor_analysis.items():
            print(f"\n{target_name.upper()} - Most Important Factors:")
            for i, (factor, importance) in enumerate(analysis['ranked_factors'], 1):
                factor_readable = {
                    'gpu_encoded': 'GPU Type',
                    'model_encoded': 'Model Architecture', 
                    'precision_encoded': 'Precision Format',
                    'batch_size': 'Batch Size'
                }.get(factor, factor)
                
                print(f"  {i}. {factor_readable}: {importance:.3f} (relative importance)")
        
        # Performance predictions
        print("\n=== PERFORMANCE FORECASTING ===")
        
        # Example predictions
        sample_configs = [
            ('NVIDIA_GeForce_RTX_4090', 'bert-large', 'prfp16', 64),
            ('NVIDIA_L40S', 'gpt2', 'prbf16', 128),
            ('Radeon_RX_7900_XT', 'resnet50', 'prfp32', 32)
        ]
        
        print("\nSample Performance Predictions:")
        for gpu, model, precision, batch_size in sample_configs:
            pred = predict_performance(regression_models, gpu, model, precision, batch_size)
            
            print(f"\n{gpu} + {model} + {precision} (batch_size={batch_size}):")
            if 'throughput' in pred and isinstance(pred['throughput'], (int, float)):
                print(f"  Predicted Throughput: {pred['throughput']:.0f} samples/sec")
            if 'latency' in pred and isinstance(pred['latency'], (int, float)):
                print(f"  Predicted Latency: {pred['latency']:.2f} ms")
            if 'memory' in pred and isinstance(pred['memory'], (int, float)):
                print(f"  Predicted Memory: {pred['memory']:.0f} MB")
        
        # Generate comprehensive predictions
        print("\n=== COMPREHENSIVE PERFORMANCE MATRIX ===")
        prediction_matrix = create_performance_predictions(regression_models, inference_df)
        
        if prediction_matrix is not None:
            # Find optimal configurations
            if 'throughput' in prediction_matrix.columns:
                optimal_configs = prediction_matrix.nlargest(5, 'throughput')
                print("\nTop 5 Predicted Configurations (by throughput):")
                for _, row in optimal_configs.iterrows():
                    print(f"  {row['gpu']} + {row['model']} + {row['precision']} "
                          f"(bs={row['batch_size']}): {row['throughput']:.0f} samples/sec")
            
            # Performance ranges by precision
            if 'throughput' in prediction_matrix.columns:
                print("\nPredicted Performance Ranges by Precision:")
                precision_stats = prediction_matrix.groupby('precision')['throughput'].agg(['min', 'max', 'mean'])
                for precision, stats in precision_stats.iterrows():
                    print(f"  {precision}: {stats['min']:.0f} - {stats['max']:.0f} samples/sec "
                          f"(avg: {stats['mean']:.0f})")
        
        # Model reliability assessment
        print("\n=== MODEL RELIABILITY ASSESSMENT ===")
        
        reliability_threshold = 0.7  # R² threshold for reliable predictions
        
        print("Model Reliability Summary:")
        for target_name, result in regression_results.items():
            test_r2 = result['test_r2']
            reliability = "HIGH" if test_r2 >= reliability_threshold else "MEDIUM" if test_r2 >= 0.5 else "LOW"
            
            print(f"  {target_name}: R² = {test_r2:.3f} - {reliability} reliability")
            
            if test_r2 < reliability_threshold:
                print(f"    Warning: Predictions may be less accurate for {target_name}")
        
        # Recommendations for model improvement
        print("\n=== MODEL IMPROVEMENT RECOMMENDATIONS ===")
        
        avg_r2 = np.mean([result['test_r2'] for result in regression_results.values()])
        
        if avg_r2 < 0.7:
            print("Recommendations to improve model accuracy:")
            print("1. Collect more diverse training data")
            print("2. Include additional features (GPU architecture details, model parameters)")
            print("3. Try non-linear regression models (Random Forest, XGBoost)")
            print("4. Feature engineering (interaction terms, polynomial features)")
        else:
            print("Models show good predictive performance!")
            print("Consider using these models for performance estimation in production")
    
    else:
        print("Could not build regression models - insufficient or invalid data")

else:
    print("No inference data available for regression analysis")

No inference data available for regression analysis


## 9. Summary and Conclusions

This section provides a comprehensive summary of all analyses conducted, key findings, and actionable recommendations for mixed-precision training and inference optimization.

In [12]:
def generate_comprehensive_summary(inference_df):
    """Generate a comprehensive summary of all analyses"""
    
    summary = {
        'dataset_overview': {},
        'performance_highlights': {},
        'efficiency_metrics': {},
        'recommendations': {}
    }
    
    if inference_df.empty:
        return "No data available for summary generation"
    
    # Dataset Overview
    summary['dataset_overview'] = {
        'total_experiments': len(inference_df),
        'unique_gpus': list(inference_df['gpu'].unique()),
        'unique_models': list(inference_df['model'].unique()),
        'unique_precisions': list(inference_df['precision'].unique()),
        'batch_size_range': f"{inference_df['batch_size'].min()} - {inference_df['batch_size'].max()}"
    }
    
    # Performance Highlights
    best_overall = inference_df.loc[inference_df['throughput_samples_per_sec'].idxmax()]
    worst_overall = inference_df.loc[inference_df['throughput_samples_per_sec'].idxmin()]
    
    summary['performance_highlights'] = {
        'best_configuration': {
            'gpu': best_overall['gpu'],
            'model': best_overall['model'],
            'precision': best_overall['precision'],
            'batch_size': best_overall['batch_size'],
            'throughput': best_overall['throughput_samples_per_sec'],
            'memory': best_overall['peak_memory_mb']
        },
        'worst_configuration': {
            'gpu': worst_overall['gpu'],
            'model': worst_overall['model'],
            'precision': worst_overall['precision'],
            'batch_size': worst_overall['batch_size'],
            'throughput': worst_overall['throughput_samples_per_sec'],
            'memory': worst_overall['peak_memory_mb']
        },
        'performance_range': {
            'max_throughput': inference_df['throughput_samples_per_sec'].max(),
            'min_throughput': inference_df['throughput_samples_per_sec'].min(),
            'avg_throughput': inference_df['throughput_samples_per_sec'].mean()
        }
    }
    
    # Efficiency Metrics
    if 'peak_memory_mb' in inference_df.columns:
        inference_df_temp = inference_df.copy()
        inference_df_temp['efficiency'] = inference_df_temp['throughput_samples_per_sec'] / inference_df_temp['peak_memory_mb']
        
        most_efficient = inference_df_temp.loc[inference_df_temp['efficiency'].idxmax()]
        summary['efficiency_metrics'] = {
            'most_memory_efficient': {
                'configuration': f"{most_efficient['gpu']} + {most_efficient['model']} + {most_efficient['precision']}",
                'efficiency': most_efficient['efficiency'],
                'throughput': most_efficient['throughput_samples_per_sec'],
                'memory': most_efficient['peak_memory_mb']
            },
            'memory_usage_stats': {
                'min_memory': inference_df['peak_memory_mb'].min(),
                'max_memory': inference_df['peak_memory_mb'].max(),
                'avg_memory': inference_df['peak_memory_mb'].mean()
            }
        }
    
    return summary

def create_executive_summary_plot(inference_df):
    """Create a comprehensive executive summary visualization"""
    
    if inference_df.empty:
        print("No data available for executive summary plot")
        return
    
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle('Mixed Precision Training & Inference: Executive Summary', fontsize=16, fontweight='bold')
    
    # 1. Performance by Precision (Box Plot)
    sns.boxplot(data=inference_df, x='precision', y='throughput_samples_per_sec', ax=axes[0,0])
    axes[0,0].set_title('Performance Distribution by Precision')
    axes[0,0].set_ylabel('Throughput (samples/sec)')
    axes[0,0].tick_params(axis='x', rotation=45)
    
    # 2. GPU Comparison (Bar Plot)
    gpu_performance = inference_df.groupby('gpu')['throughput_samples_per_sec'].mean().sort_values(ascending=False)
    gpu_performance.plot(kind='bar', ax=axes[0,1])
    axes[0,1].set_title('Average Performance by GPU')
    axes[0,1].set_ylabel('Avg Throughput (samples/sec)')
    axes[0,1].tick_params(axis='x', rotation=45)
    
    # 3. Memory vs Performance Scatter
    sns.scatterplot(data=inference_df, x='peak_memory_mb', y='throughput_samples_per_sec',
                   hue='precision', style='gpu', s=100, ax=axes[0,2])
    axes[0,2].set_title('Memory vs Performance Trade-off')
    axes[0,2].set_xlabel('Peak Memory (MB)')
    axes[0,2].set_ylabel('Throughput (samples/sec)')
    
    # 4. Model Performance Comparison
    model_stats = inference_df.groupby('model')['throughput_samples_per_sec'].agg(['mean', 'std']).reset_index()
    axes[1,0].bar(model_stats['model'], model_stats['mean'], yerr=model_stats['std'], capsize=5)
    axes[1,0].set_title('Model Performance Comparison')
    axes[1,0].set_ylabel('Throughput (samples/sec)')
    axes[1,0].tick_params(axis='x', rotation=45)
    
    # 5. Precision Format Efficiency
    if 'peak_memory_mb' in inference_df.columns:
        inference_df_temp = inference_df.copy()
        inference_df_temp['efficiency'] = inference_df_temp['throughput_samples_per_sec'] / inference_df_temp['peak_memory_mb']
        efficiency_by_precision = inference_df_temp.groupby('precision')['efficiency'].mean().sort_values(ascending=False)
        efficiency_by_precision.plot(kind='bar', ax=axes[1,1], color='green', alpha=0.7)
        axes[1,1].set_title('Memory Efficiency by Precision')
        axes[1,1].set_ylabel('Efficiency (samples/sec/MB)')
        axes[1,1].tick_params(axis='x', rotation=45)
    
    # 6. Performance Improvement Heatmap
    improvement_matrix = inference_df.pivot_table(
        values='throughput_samples_per_sec', 
        index='model', 
        columns='precision', 
        aggfunc='mean'
    )
    
    # Calculate improvement relative to FP32
    if 'prfp32' in improvement_matrix.columns:
        for col in improvement_matrix.columns:
            if col != 'prfp32':
                improvement_matrix[col] = ((improvement_matrix[col] - improvement_matrix['prfp32']) / improvement_matrix['prfp32']) * 100
        
        # Remove FP32 column as it would be all zeros
        improvement_matrix = improvement_matrix.drop('prfp32', axis=1)
    
    sns.heatmap(improvement_matrix, annot=True, fmt='.1f', cmap='RdYlGn', center=0, ax=axes[1,2])
    axes[1,2].set_title('Performance Improvement vs FP32 (%)')
    
    plt.tight_layout()
    plt.show()

def print_key_findings(inference_df):
    """Print key findings and insights"""
    
    if inference_df.empty:
        print("No data available for key findings")
        return
    
    print("=" * 80)
    print("KEY FINDINGS AND INSIGHTS")
    print("=" * 80)
    
    # 1. Best overall performance
    best_config = inference_df.loc[inference_df['throughput_samples_per_sec'].idxmax()]
    print(f"\n🏆 BEST OVERALL PERFORMANCE:")
    print(f"   Configuration: {best_config['gpu']} + {best_config['model']} + {best_config['precision']}")
    print(f"   Throughput: {best_config['throughput_samples_per_sec']:.0f} samples/sec")
    print(f"   Memory Usage: {best_config['peak_memory_mb']:.0f} MB")
    print(f"   Batch Size: {best_config['batch_size']}")
    
    # 2. Precision format analysis
    precision_performance = inference_df.groupby('precision')['throughput_samples_per_sec'].agg(['mean', 'std', 'count'])
    print(f"\n📊 PRECISION FORMAT PERFORMANCE:")
    for precision, stats in precision_performance.iterrows():
        print(f"   {precision}: {stats['mean']:.0f} ± {stats['std']:.0f} samples/sec (n={stats['count']})")
    
    # 3. GPU comparison
    gpu_performance = inference_df.groupby('gpu')['throughput_samples_per_sec'].mean().sort_values(ascending=False)
    print(f"\n🖥️  GPU PERFORMANCE RANKING:")
    for i, (gpu, performance) in enumerate(gpu_performance.items(), 1):
        print(f"   {i}. {gpu}: {performance:.0f} samples/sec (average)")
    
    # 4. Model-specific insights
    print(f"\n🤖 MODEL-SPECIFIC INSIGHTS:")
    for model in inference_df['model'].unique():
        model_data = inference_df[inference_df['model'] == model]
        best_precision = model_data.loc[model_data['throughput_samples_per_sec'].idxmax(), 'precision']
        best_performance = model_data['throughput_samples_per_sec'].max()
        print(f"   {model}: Best with {best_precision} ({best_performance:.0f} samples/sec)")
    
    # 5. Memory efficiency
    if 'peak_memory_mb' in inference_df.columns:
        inference_df_temp = inference_df.copy()
        inference_df_temp['efficiency'] = inference_df_temp['throughput_samples_per_sec'] / inference_df_temp['peak_memory_mb']
        most_efficient = inference_df_temp.loc[inference_df_temp['efficiency'].idxmax()]
        
        print(f"\n💾 MEMORY EFFICIENCY:")
        print(f"   Most efficient: {most_efficient['gpu']} + {most_efficient['model']} + {most_efficient['precision']}")
        print(f"   Efficiency: {most_efficient['efficiency']:.3f} samples/sec/MB")
        
        # Memory savings
        fp32_memory = inference_df[inference_df['precision'] == 'prfp32']['peak_memory_mb'].mean()
        other_precisions = inference_df[inference_df['precision'] != 'prfp32']
        
        if not other_precisions.empty and not pd.isna(fp32_memory):
            print(f"\n   Memory Savings vs FP32:")
            for precision in other_precisions['precision'].unique():
                precision_memory = other_precisions[other_precisions['precision'] == precision]['peak_memory_mb'].mean()
                savings = ((fp32_memory - precision_memory) / fp32_memory) * 100
                print(f"     {precision}: {savings:.1f}% memory savings")
    
    # 6. Statistical significance
    precision_groups = [group['throughput_samples_per_sec'].values for name, group in inference_df.groupby('precision')]
    if len(precision_groups) >= 2:
        try:
            f_stat, p_value = stats.f_oneway(*precision_groups)
            significance = "statistically significant" if p_value < 0.05 else "not statistically significant"
            print(f"\n📈 STATISTICAL ANALYSIS:")
            print(f"   Precision format differences are {significance} (p={p_value:.4f})")
        except:
            print(f"\n📈 STATISTICAL ANALYSIS: Could not perform ANOVA test")

def generate_recommendations():
    """Generate actionable recommendations"""
    
    print("\n" + "=" * 80)
    print("ACTIONABLE RECOMMENDATIONS")
    print("=" * 80)
    
    recommendations = [
        {
            "category": "🎯 Production Deployment",
            "items": [
                "Use FP16 or BF16 for inference when accuracy requirements allow",
                "Optimize batch sizes for each GPU-model combination",
                "Monitor memory usage to avoid OOM errors",
                "Implement dynamic batch sizing for variable input sizes"
            ]
        },
        {
            "category": "🔧 Performance Optimization", 
            "items": [
                "Profile GPU utilization to identify bottlenecks",
                "Use tensor cores when available (FP16/BF16 on modern GPUs)",
                "Consider model parallelism for large models",
                "Implement efficient data loading to maximize GPU utilization"
            ]
        },
        {
            "category": "💰 Cost-Effectiveness",
            "items": [
                "Lower precision formats can reduce memory requirements",
                "Higher throughput enables processing more data with same hardware",
                "Energy efficiency improvements reduce operational costs",
                "Memory savings allow larger batch sizes or concurrent processes"
            ]
        },
        {
            "category": "⚠️ Risk Mitigation",
            "items": [
                "Validate model accuracy when switching precision formats",
                "Implement fallback to FP32 for numerical instability",
                "Monitor for gradient underflow in mixed precision training",
                "Test precision formats thoroughly before production deployment"
            ]
        },
        {
            "category": "🔬 Future Research",
            "items": [
                "Investigate INT8 quantization for further optimization", 
                "Explore dynamic precision switching during training",
                "Study model architecture impact on precision sensitivity",
                "Develop automated precision selection algorithms"
            ]
        }
    ]
    
    for rec in recommendations:
        print(f"\n{rec['category']}:")
        for item in rec['items']:
            print(f"   • {item}")

# Generate comprehensive summary
print("=" * 80)
print("COMPREHENSIVE ANALYSIS SUMMARY")
print("=" * 80)

if not inference_df.empty:
    
    # Generate summary statistics
    summary_data = generate_comprehensive_summary(inference_df)
    
    if isinstance(summary_data, dict):
        print(f"\n📋 DATASET OVERVIEW:")
        overview = summary_data['dataset_overview']
        print(f"   Total Experiments: {overview['total_experiments']}")
        print(f"   GPUs Tested: {', '.join(overview['unique_gpus'])}")
        print(f"   Models Tested: {', '.join(overview['unique_models'])}")
        print(f"   Precision Formats: {', '.join(overview['unique_precisions'])}")
        print(f"   Batch Size Range: {overview['batch_size_range']}")
    
    # Print key findings
    print_key_findings(inference_df)
    
    # Create executive summary visualization
    print(f"\nCreating executive summary visualization...")
    create_executive_summary_plot(inference_df)
    
    # Generate recommendations
    generate_recommendations()
    
    # Final insights
    print(f"\n" + "=" * 80)
    print("THESIS CONTRIBUTION SUMMARY")
    print("=" * 80)
    
    contributions = [
        "🔍 Comprehensive analysis of mixed precision training across multiple architectures",
        "📊 Statistical validation of performance improvements with confidence intervals",
        "⚡ Quantified energy efficiency gains from reduced precision formats",
        "🎯 Hardware-specific optimization recommendations for NVIDIA and AMD GPUs",
        "📈 Regression models for performance prediction across configurations",
        "💾 Memory usage optimization strategies for resource-constrained environments",
        "🛠️ Practical batch size optimization algorithms",
        "📋 Systematic methodology for mixed precision evaluation"
    ]
    
    print(f"\nThis thesis provides:")
    for contribution in contributions:
        print(f"   {contribution}")
    
    print(f"\n💡 KEY TAKEAWAY:")
    print(f"   Mixed precision training offers significant performance and efficiency")
    print(f"   improvements with minimal accuracy trade-offs when properly configured.")
    print(f"   The optimal configuration depends on the specific combination of hardware,")
    print(f"   model architecture, and application requirements.")
    
else:
    print("❌ No experimental data available for comprehensive analysis.")
    print("Please ensure the data files are properly loaded before running this analysis.")

print(f"\n🎉 Analysis complete! All results are ready for thesis integration.")
print("=" * 80)

COMPREHENSIVE ANALYSIS SUMMARY
❌ No experimental data available for comprehensive analysis.
Please ensure the data files are properly loaded before running this analysis.

🎉 Analysis complete! All results are ready for thesis integration.
